In [1]:
import mlflow

mlflow.set_experiment("Credit Card Fraud Detection")

print("MLflow experiment configured.")

MLflow experiment configured.


In [2]:
mlflow.set_experiment("Credit Card Fraud Detection")

print("Experiment ready.")

Experiment ready.


In [3]:
with mlflow.start_run():
    print("MLflow run started.")

MLflow run started.


In [4]:
with mlflow.start_run():
    mlflow.log_params({
        "model":"XGBoost",
        "n_estimators":200,
        "max_depth":6,
        "learning_rate":0.1,
        "subsample":0.8,
        "colsample_bytree":0.8,
        "scale_pos_weight":599.4761904761905,
    })

    print("Paramaters logged")

Paramaters logged


In [5]:
with mlflow.start_run():
    mlflow.log_metrics({
        "precision":0.9048,
        "recall":0.8000,
        "f1_score":0.8492,
        "roc_auc":0.9761,
        "pr_auc":0.8248,
    })

    print("Metrics logged. ")

Metrics logged. 


In [6]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



In [7]:
X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")

X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")
y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(226980, 30) (226980,)
(56746, 30) (56746,)


In [8]:
scale_pos_weight = (y_train == 0).sum()/(y_train == 1).sum()

model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1,
)

print(f"scale_pos_weight: {scale_pos_weight}")

scale_pos_weight: 599.4761904761905


In [9]:
with mlflow.start_run(run_name = 'Final XGBoost baseline'):

    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]

    threshold = 0.2325

    y_pred = (y_proba >= threshold).astype(int)

    mlflow.log_params({
        "model":"XGBoost",
        "n_estimators":200,
        "max_depth":6,
        "learning_rate":0.1,
        "subsample":0.6,
        "colsample_bytree":0.6,
        "scale_pos_weight":scale_pos_weight,
        "threshold":threshold,
    })

    mlflow.log_metrics({
        "precision":precision_score(y_test,y_pred),
        "recall":recall_score(y_test,y_pred),
        "f1_score":f1_score(y_test,y_pred),
        "roc_auc":roc_auc_score(y_test,y_pred),
        "pr_auc":average_precision_score(y_test,y_pred)
    })

    print("Training and experiment tracking completed.")



Training and experiment tracking completed.
